In [ ]:
import ROOT
%load_ext JupyROOT

In [ ]:
%%cpp
const int NTRIALS = 10000;
const int NPART = 40;

In [ ]:
%%cpp
  gDirectory->Clear();
  TObject *obj = gROOT->FindObject("cPoisson");
  if(obj)
  {
      delete obj;
  }

  gStyle->SetOptStat(0);
  TRandom3 *rand = new TRandom3(0);

  TH1F *PoissonBins[10];
  TF1 *poisson_func[10];

  for(int i = 0; i < 10; i++)
  {
    PoissonBins[i] = new TH1F(Form("PoissonBins_%d", i), Form("Bin %d;Number of particles;Probability Density", i+1), 20, -0.5, 19.5);
    PoissonBins[i]->Sumw2();
    poisson_func[i] = new TF1(Form("poisson_func_%d", i), "[0]*TMath::Poisson(x, [1])", 0, 20);
    poisson_func[i]->SetParameters(1., 4.);
  }

  for(int n = 0; n < NTRIALS; n++)
  {
    TH1F *GenEnergy = new TH1F("GenEnergy", "Particle Energy;Energy (GeV);Entries", 10, 0., 10.);
    GenEnergy->Sumw2();
    for(int i = 0; i < NPART; i++)
    {
      float energy = rand->Uniform(0., 10.);
      GenEnergy->Fill(energy);
    }

    for(int i = 1; i <= GenEnergy->GetXaxis()->GetNbins(); i++)
    {
      PoissonBins[i-1]->Fill(GenEnergy->GetBinContent(i));
    }

    delete GenEnergy;
  }

  TCanvas *cPoisson = new TCanvas("cPoisson", "cPoisson", 1000, 700);
  cPoisson->Divide(5,2);
  for(int i = 1; i <= 10; i++)
  {
    cPoisson->cd(i);
    gPad->SetLeftMargin(0.15);
    gPad->SetBottomMargin(0.15);
    PoissonBins[i-1]->Scale(1./PoissonBins[i-1]->Integral());
    PoissonBins[i-1]->Fit(poisson_func[i-1], "RL", "", 0., 12.);
    PoissonBins[i-1]->SetLineColor(kBlack);
    PoissonBins[i-1]->SetLineWidth(2);
    gStyle->SetTitleFontSize(0.08);
    PoissonBins[i-1]->GetYaxis()->SetTitleSize(0.06);
    PoissonBins[i-1]->GetXaxis()->SetTitleSize(0.06);
    PoissonBins[i-1]->Draw("");
    poisson_func[i-1]->Draw("same");
    TLatex *text = new TLatex(0.6, 0.8, Form("#lambda = %.2lf", poisson_func[i-1]->GetParameter(1)));
    text->SetNDC();
    text->SetTextSize(0.08);
    text->SetTextColor(kBlack);
    text->Draw("same");
  }
  cPoisson->Draw();